In [1]:
# dfcl数据准备，统计

In [2]:
from pathlib import Path
import sys
import os
import json
import math
import datetime
import warnings
from datetime import datetime as dt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import joblib
import xgboost as xgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve,
)
import findspark
findspark.init()

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as func
from pyspark.sql.functions import col, desc, udf
from pyspark.sql.types import StructType, StructField, StringType

from ortools.linear_solver import pywraplp

warnings.filterwarnings("ignore")


# Backflow imports
from backflow import exp_setting
from backflow.programming import pgt
from backflow.calib import metrics as calib_metric
from backflow.calib import stats as calib_stats
from backflow.visualization import vis
from backflow.programming import simulation as pgtsimu
from backflow.utils import tools as bkfl_tools


# Matplotlib font

from pyspark.sql import SparkSession
spark3conf_best = {
    "spark.submit.deployMode": "client",
    "spark.yarn.queue": "root.growstrategy_group_prod",

    "spark.driver.memory": "24g",
    "spark.driver.memoryOverhead": "4g",

    "spark.dynamicAllocation.enabled": "true",
    "spark.dynamicAllocation.minExecutors": "80",
    "spark.dynamicAllocation.initialExecutors": "80",
    "spark.dynamicAllocation.maxExecutors": "120",
    "spark.dynamicAllocation.shuffleTracking.enabled": "true",

    "spark.executor.cores": "2",
    "spark.executor.memory": "12g",
    "spark.executor.memoryOverhead": "8g",

    "spark.default.parallelism": "1200",
    "spark.sql.shuffle.partitions": "1200",

    "spark.sql.adaptive.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.enabled": "true",
    "spark.sql.adaptive.skewJoin.enabled": "true",
    "spark.sql.adaptive.advisoryPartitionSizeInBytes": "134217728",

    "spark.sql.autoBroadcastJoinThreshold": "104857600",
    "spark.sql.broadcastTimeout": "3000",

    "spark.network.timeout": "600s",
    "spark.executor.heartbeatInterval": "60s",

    "spark.memory.fraction": "0.8",
    "spark.memory.storageFraction": "0.3",

    "spark.sql.execution.arrow.pyspark.enabled": "true",
    "spark.sql.execution.arrow.maxRecordsPerBatch": "5000",

    "mapred.input.dir.recursive": "TRUE",
    "hive.mapred.supports.subdirectories": "TRUE",

    "spark.serializer": "org.apache.spark.serializer.KryoSerializer",
    "spark.kryoserializer.buffer.max": "1024m",

    "spark.yarn.dist.archives": "hdfs://difed/user/dm/ENV/dlflow/v2/py3.7.15_for_tfcluster_63.tgz",
    "spark.executorEnv.PYSPARK_PYTHON": "./py3.7.15_for_tfcluster_63.tgz/py3.7.15_for_tfcluster/bin/python3",

    "spark.extraListeners": "sparkmonitor.listener.JupyterSparkMonitorListener",

    "spark.sql.merge.mode": "FAST",
    "spark.sql.merge.enabled": "TRUE",
}

builder = (
    SparkSession
    .builder
    .appName("flowback_euen_v22")
    .master("yarn")
)

for k, v in spark3conf_best.items():
    builder = builder.config(k, v)

spark = builder.enableHiveSupport().getOrCreate()

print("[Info] Spark Version:", spark.version)
print("[Info] Spark Queue:", spark.conf.get("spark.yarn.queue"))
print("[Info] Spark App ID:", spark.sparkContext.applicationId)
print(spark.conf.get("spark.yarn.queue"))
print(spark.sparkContext.applicationId)

load metric func from causalml...


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
Application Id: application_1782979140233_77052061, Tracking URL: http://bigdata-nmg-hdprm00.nmg01:8088/proxy/application_1782979140233_77052061/
INFO:SparkMonitorKernel:Client Connected ('127.0.0.1', 47370)


[Info] Spark Version: 3.2.0-101
[Info] Spark Queue: root.growstrategy_group_prod
[Info] Spark App ID: application_1782979140233_77052061
root.growstrategy_group_prod
application_1782979140233_77052061


In [2]:
query = """
with base as (
    select distinct
        dt,
        pid,
        case
            when flowtree_group_name = 'DS空白'
             and group_name_eve = 'control_group'
                then 'DS空白'
            else group_name_eve
        end as treatment_name
    from dm.ut_alg_flowback_msg_outcall_label_v2_di
    where dt between '2026-05-11' and '2026-08-21'
      and dt not between '2026-06-18' and '2026-06-22'
      and dt not in ('2026-08-02', '2026-08-08', '2026-08-09')
      and new_exp = '迭代流量'
      and pid is not null
      and (
          group_name_eve in (
              '留存6张3元',
              '8元留存5张3',
              '留存6张5元',
              '10元留存5张3'
          )
          or (
              flowtree_group_name = 'DS空白'
              and group_name_eve = 'control_group'
          )
      )
),

ordered_data as (
    select
        dt,
        pid,
        treatment_name,
        lag(dt) over (
            partition by pid, treatment_name
            order by dt
        ) as previous_dt
    from base
),

flag_data as (
    select
        *,
        case
            when previous_dt is not null
             and datediff(to_date(dt), to_date(previous_dt)) between 1 and 6
                then 1
            else 0
        end as repeat_7d_flag
    from ordered_data
)

select
    count(*) as total_record_cnt,
    sum(repeat_7d_flag) as repeat_7d_record_cnt,
    sum(repeat_7d_flag) / count(*) as repeat_7d_record_ratio,

    count(distinct pid) as total_user_cnt,
    count(distinct case when repeat_7d_flag = 1 then pid end)
        as repeat_7d_user_cnt,
    count(distinct case when repeat_7d_flag = 1 then pid end)
        / count(distinct pid) as repeat_7d_user_ratio
from flag_data
"""

In [4]:
# 用户7天内时候重复入主
result_pdf = spark.sql(query).toPandas()
result_pdf

,total_record_cnt,repeat_7d_record_cnt,repeat_7d_record_ratio,total_user_cnt,repeat_7d_user_cnt,repeat_7d_user_ratio
0,12031275,7627613,0.633982,1882830,1155168,0.613528


In [5]:
# 用户是否跨组
query = """
with base as (
    select distinct
        dt,
        pid,
        case
            when flowtree_group_name = 'DS空白'
             and group_name_eve = 'control_group' then 0
            when group_name_eve = '留存6张3元' then 1
            when group_name_eve = '8元留存5张3' then 2
            when group_name_eve = '留存6张5元' then 3
            when group_name_eve = '10元留存5张3' then 4
        end as treatment_id
    from dm.ut_alg_flowback_msg_outcall_label_v2_di
    where dt between '2026-05-11' and '2026-08-21'
      and dt not between '2026-06-18' and '2026-06-22'
      and dt not in ('2026-08-02', '2026-08-08', '2026-08-09')
      and new_exp = '迭代流量'
      and pid is not null
      and (
          group_name_eve in (
              '留存6张3元',
              '8元留存5张3',
              '留存6张5元',
              '10元留存5张3'
          )
          or (
              flowtree_group_name = 'DS空白'
              and group_name_eve = 'control_group'
          )
      )
),

ordered_data as (
    select
        dt,
        pid,
        treatment_id,
        lag(dt) over (
            partition by pid
            order by dt, treatment_id
        ) as previous_dt,
        lag(treatment_id) over (
            partition by pid
            order by dt, treatment_id
        ) as previous_treatment_id
    from base
),

flag_data as (
    select
        *,
        datediff(to_date(dt), to_date(previous_dt)) as gap_days,
        case
            when previous_dt is not null
             and datediff(to_date(dt), to_date(previous_dt)) between 0 and 6
             and treatment_id <> previous_treatment_id
                then 1
            else 0
        end as cross_group_7d_flag,
        case
            when previous_dt is not null
             and datediff(to_date(dt), to_date(previous_dt)) = 0
             and treatment_id <> previous_treatment_id
                then 1
            else 0
        end as cross_group_same_day_flag
    from ordered_data
)

select
    count(distinct pid) as total_user_cnt,

    sum(cross_group_7d_flag) as cross_group_7d_record_cnt,
    count(distinct case when cross_group_7d_flag = 1 then pid end)
        as cross_group_7d_user_cnt,
    cast(
        count(distinct case when cross_group_7d_flag = 1 then pid end)
        as double
    ) / count(distinct pid) as cross_group_7d_user_ratio,

    sum(cross_group_same_day_flag) as cross_group_same_day_record_cnt,
    count(distinct case when cross_group_same_day_flag = 1 then pid end)
        as cross_group_same_day_user_cnt
from flag_data
"""

spark.sql(query).toPandas()

,total_user_cnt,cross_group_7d_record_cnt,cross_group_7d_user_cnt,cross_group_7d_user_ratio,cross_group_same_day_record_cnt,cross_group_same_day_user_cnt
0,1882830,0,0,0.0,0,0


In [6]:
# 每日分流比例

query = """
with base as (
    select distinct
        dt,
        pid,
        case
            when flowtree_group_name = 'DS空白'
             and group_name_eve = 'control_group' then 0
            when group_name_eve = '留存6张3元' then 1
            when group_name_eve = '8元留存5张3' then 2
            when group_name_eve = '留存6张5元' then 3
            when group_name_eve = '10元留存5张3' then 4
        end as treatment_id,
        case
            when flowtree_group_name = 'DS空白'
             and group_name_eve = 'control_group' then 'DS空白'
            else group_name_eve
        end as treatment_name
    from dm.ut_alg_flowback_msg_outcall_label_v2_di
    where dt between '2026-05-11' and '2026-08-21'
      and dt not between '2026-06-18' and '2026-06-22'
      and dt not in ('2026-08-02', '2026-08-08', '2026-08-09')
      and new_exp = '迭代流量'
      and pid is not null
      and (
          group_name_eve in (
              '留存6张3元',
              '8元留存5张3',
              '留存6张5元',
              '10元留存5张3'
          )
          or (
              flowtree_group_name = 'DS空白'
              and group_name_eve = 'control_group'
          )
      )
),

daily_group_count as (
    select
        dt,
        treatment_id,
        treatment_name,
        count(*) as group_user_cnt
    from base
    group by dt, treatment_id, treatment_name
),

daily_propensity as (
    select
        dt,
        treatment_id,
        treatment_name,
        group_user_cnt,
        sum(group_user_cnt) over (partition by dt) as total_user_cnt
    from daily_group_count
),

window_group_count as (
    select
        treatment_id,
        treatment_name,
        count(*) as group_user_day_cnt
    from base
    group by treatment_id, treatment_name
),

window_propensity as (
    select
        treatment_id,
        treatment_name,
        group_user_day_cnt,
        sum(group_user_day_cnt) over () as total_user_day_cnt
    from window_group_count
)

-- 每日分流概率
select
    'daily' as scope,
    dt,
    treatment_id,
    treatment_name,
    group_user_cnt as group_cnt,
    total_user_cnt as total_cnt,
    cast(group_user_cnt as double) / total_user_cnt as propensity
from daily_propensity


order by scope, dt, treatment_id
"""

propensity_pdf = spark.sql(query).toPandas()
propensity_pdf

,scope,dt,treatment_id,treatment_name,group_cnt,total_cnt,propensity
0,daily,2026-05-11,0,DS空白,52422,106944,0.490182
1,daily,2026-05-11,1,留存6张3元,18964,106944,0.177326
2,daily,2026-05-11,2,8元留存5张3,12693,106944,0.118688
3,daily,2026-05-11,3,留存6张5元,9922,106944,0.092778
4,daily,2026-05-11,4,10元留存5张3,12943,106944,0.121026
...,...,...,...,...,...,...,...
470,daily,2026-08-21,0,DS空白,67004,134500,0.498171
471,daily,2026-08-21,1,留存6张3元,23621,134500,0.175621
472,daily,2026-08-21,2,8元留存5张3,15717,134500,0.116855
473,daily,2026-08-21,3,留存6张5元,12396,134500,0.092164


In [7]:
print(propensity_pdf.to_string())

     scope          dt  treatment_id treatment_name  group_cnt  total_cnt  propensity
0    daily  2026-05-11             0           DS空白      52422     106944    0.490182
1    daily  2026-05-11             1         留存6张3元      18964     106944    0.177326
2    daily  2026-05-11             2        8元留存5张3      12693     106944    0.118688
3    daily  2026-05-11             3         留存6张5元       9922     106944    0.092778
4    daily  2026-05-11             4       10元留存5张3      12943     106944    0.121026
5    daily  2026-05-12             0           DS空白      53009     107635    0.492489
6    daily  2026-05-12             1         留存6张3元      18952     107635    0.176077
7    daily  2026-05-12             2        8元留存5张3      12620     107635    0.117248
8    daily  2026-05-12             3         留存6张5元      10025     107635    0.093139
9    daily  2026-05-12             4       10元留存5张3      13029     107635    0.121048
10   daily  2026-05-13             0           DS空白   

In [8]:
query = """
with base as (
    -- 第一步：同一天、同一用户、同一组只保留一条
    select distinct
        dt,
        pid,
        case
            when flowtree_group_name = 'DS空白'
             and group_name_eve = 'control_group' then 0
            when group_name_eve = '留存6张3元' then 1
            when group_name_eve = '8元留存5张3' then 2
            when group_name_eve = '留存6张5元' then 3
            when group_name_eve = '10元留存5张3' then 4
        end as treatment_id,
        case
            when flowtree_group_name = 'DS空白'
             and group_name_eve = 'control_group' then 'DS空白'
            else group_name_eve
        end as treatment_name
    from dm.ut_alg_flowback_msg_outcall_label_v2_di
    where dt between '2026-05-11' and '2026-08-21'
      and dt not between '2026-06-18' and '2026-06-22'
      and dt not in ('2026-08-02', '2026-08-08', '2026-08-09')
      and new_exp = '迭代流量'
      and pid is not null
      and (
          group_name_eve in (
              '留存6张3元',
              '8元留存5张3',
              '留存6张5元',
              '10元留存5张3'
          )
          or (
              flowtree_group_name = 'DS空白'
              and group_name_eve = 'control_group'
          )
      )
),

ordered_data as (
    select
        dt,
        pid,
        treatment_id,
        treatment_name,
        lag(dt) over (
            partition by pid
            order by dt, treatment_id
        ) as previous_dt
    from base
),

entry_flag as (
    select
        *,
        case
            when previous_dt is null then 1
            when datediff(to_date(dt), to_date(previous_dt)) >= 7 then 1
            else 0
        end as is_new_entry
    from ordered_data
),

-- 每个连续的7天周期只保留第一条
valid_entry as (
    select
        dt as entry_dt,
        pid,
        treatment_id,
        treatment_name
    from entry_flag
    where is_new_entry = 1
),

daily_group_count as (
    select
        entry_dt,
        treatment_id,
        treatment_name,
        count(*) as group_entry_cnt
    from valid_entry
    group by entry_dt, treatment_id, treatment_name
),

daily_propensity as (
    select
        entry_dt,
        treatment_id,
        treatment_name,
        group_entry_cnt,
        sum(group_entry_cnt) over (
            partition by entry_dt
        ) as total_entry_cnt
    from daily_group_count
),

window_group_count as (
    select
        treatment_id,
        treatment_name,
        count(*) as group_entry_cnt
    from valid_entry
    group by treatment_id, treatment_name
),

window_propensity as (
    select
        treatment_id,
        treatment_name,
        group_entry_cnt,
        sum(group_entry_cnt) over () as total_entry_cnt
    from window_group_count
)

select
    'daily' as scope,
    entry_dt as dt,
    treatment_id,
    treatment_name,
    group_entry_cnt as group_cnt,
    total_entry_cnt as total_cnt,
    cast(group_entry_cnt as double) / total_entry_cnt as propensity
from daily_propensity

union all

select
    'window' as scope,
    'ALL' as dt,
    treatment_id,
    treatment_name,
    group_entry_cnt as group_cnt,
    total_entry_cnt as total_cnt,
    cast(group_entry_cnt as double) / total_entry_cnt as propensity
from window_propensity

order by scope, dt, treatment_id
"""

entry_propensity_pdf = spark.sql(query).toPandas()
entry_propensity_pdf

,scope,dt,treatment_id,treatment_name,group_cnt,total_cnt,propensity
0,daily,2026-05-11,0,DS空白,52422,106944,0.490182
1,daily,2026-05-11,1,留存6张3元,18964,106944,0.177326
2,daily,2026-05-11,2,8元留存5张3,12693,106944,0.118688
3,daily,2026-05-11,3,留存6张5元,9922,106944,0.092778
4,daily,2026-05-11,4,10元留存5张3,12943,106944,0.121026
...,...,...,...,...,...,...,...
475,window,ALL,0,DS空白,2164815,4403662,0.491594
476,window,ALL,1,留存6张3元,782878,4403662,0.177779
477,window,ALL,2,8元留存5张3,521821,4403662,0.118497
478,window,ALL,3,留存6张5元,411671,4403662,0.093484


In [9]:
# 查看euen训练样本

audit_detail_df = spark.sql("""
with feature_data as (
    select
        flowtree_group_name,
        group_name_eve,
        dt,
        pid
    from dm.ut_alg_flowback_msg_outcall_feature_train_di
    where dt between '2026-05-11' and '2026-08-21'
      and dt not between '2026-06-18' and '2026-06-22'
      and dt not in ('2026-08-02', '2026-08-08', '2026-08-09')
      and new_exp = '迭代流量'
      and pid is not null
      and (
          group_name_eve in (
              '留存6张3元',
              '8元留存5张3',
              '留存6张5元',
              '10元留存5张3'
          )
          or (
              flowtree_group_name = 'DS空白'
              and group_name_eve = 'control_group'
          )
      )
),

label_data as (
    select
        flowtree_group_name,
        group_name_eve,
        dt,
        pid,
        finish_cnt_7d
    from dm.ut_alg_flowback_msg_outcall_label_v2_di
    where dt between '2026-05-11' and '2026-08-21'
      and dt not between '2026-06-18' and '2026-06-22'
      and dt not in ('2026-08-02', '2026-08-08', '2026-08-09')
      and new_exp = '迭代流量'
      and pid is not null
      and (
          group_name_eve in (
              '留存6张3元',
              '8元留存5张3',
              '留存6张5元',
              '10元留存5张3'
          )
          or (
              flowtree_group_name = 'DS空白'
              and group_name_eve = 'control_group'
          )
      )
),

joined_data as (
    select
        f.pid,
        f.dt,
        f.flowtree_group_name,
        f.group_name_eve,
        l.finish_cnt_7d,
        sum(l.finish_cnt_7d) over (
            partition by f.pid
        ) as finish_all_cnt
    from feature_data f
    join label_data l
      on f.pid = l.pid
     and f.dt = l.dt
     and f.flowtree_group_name = l.flowtree_group_name
     and f.group_name_eve = l.group_name_eve
),

-- 完全复现旧 SQL 最后的 outcome 筛选
old_train_rows as (
    select *
    from joined_data
    where
        (finish_all_cnt > 0 and finish_cnt_7d > 0)
        or finish_all_cnt = 0
),

-- 检查同一个 pid+dt+组在最终数据中是否仍有多行
key_stats as (
    select
        pid,
        dt,
        group_name_eve,
        count(*) as key_row_cnt,
        count(distinct finish_cnt_7d) as label_value_cnt,
        max(finish_cnt_7d) as finish_cnt_7d
    from old_train_rows
    group by pid, dt, group_name_eve
),

ordered_data as (
    select
        *,
        lag(dt) over (
            partition by pid, group_name_eve
            order by dt
        ) as previous_dt,
        lag(finish_cnt_7d) over (
            partition by pid, group_name_eve
            order by dt
        ) as previous_finish_cnt_7d
    from key_stats
)

select
    *,
    datediff(to_date(dt), to_date(previous_dt)) as gap_days,
    case
        when previous_dt is not null
         and datediff(to_date(dt), to_date(previous_dt)) between 1 and 6
            then 1
        else 0
    end as repeat_7d_flag
from ordered_data
""")

audit_detail_df.createOrReplaceTempView(
    "v5_euen_train_audit_detail"
)

In [10]:
# 统计重复情况
# exact_duplicate_row_cnt：同一天、同用户、同组是否因为源表或 join 产生多行；
# repeat_7d_record_cnt：旧 EUEN 最终训练数据中，有多少跨日期同组重复记录；
# repeat_7d_user_cnt：涉及多少用户；
# repeat_7d_label_changed_cnt：同一用户同组的连续日期记录，其7日 label 是否发生变化。

summary_pdf = spark.sql("""
select
    sum(key_row_cnt) as old_train_total_rows,

    count(*) as distinct_pid_dt_group_rows,

    sum(key_row_cnt - 1) as exact_duplicate_row_cnt,

    sum(
        case when label_value_cnt > 1 then 1 else 0 end
    ) as same_day_label_conflict_cnt,

    sum(repeat_7d_flag) as repeat_7d_record_cnt,

    count(distinct case
        when repeat_7d_flag = 1 then pid
    end) as repeat_7d_user_cnt,

    cast(sum(repeat_7d_flag) as double)
        / count(*) as repeat_7d_record_ratio,

    sum(case
        when repeat_7d_flag = 1
         and finish_cnt_7d <> previous_finish_cnt_7d
        then 1 else 0
    end) as repeat_7d_label_changed_cnt
from v5_euen_train_audit_detail
""").toPandas()

summary_pdf

,old_train_total_rows,distinct_pid_dt_group_rows,exact_duplicate_row_cnt,same_day_label_conflict_cnt,repeat_7d_record_cnt,repeat_7d_user_cnt,repeat_7d_record_ratio,repeat_7d_label_changed_cnt
0,11429795,11429795,0,0,7147126,1082389,0.625307,2038


In [12]:
example_pdf = spark.sql("""
select
    pid,
    previous_dt,
    dt,
    group_name_eve,
    gap_days,
    previous_finish_cnt_7d,
    finish_cnt_7d,
    key_row_cnt
from v5_euen_train_audit_detail
where repeat_7d_flag = 1
and previous_finish_cnt_7d <> finish_cnt_7d
order by pid, dt
limit 100
""").toPandas()

example_pdf

,pid,previous_dt,dt,group_name_eve,gap_days,previous_finish_cnt_7d,finish_cnt_7d,key_row_cnt
0,100500693,2026-07-06,2026-07-08,留存6张5元,2,3.0,1.0,1
1,101134591,2026-08-03,2026-08-05,10元留存5张3,2,8.0,3.0,1
2,1011701317632,2026-07-06,2026-07-08,10元留存5张3,2,3.0,1.0,1
3,1016407531520,2026-08-01,2026-08-04,留存6张5元,3,4.0,2.0,1
4,102213398,2026-05-24,2026-05-29,留存6张5元,5,1.0,2.0,1
...,...,...,...,...,...,...,...,...
95,145375595,2026-05-28,2026-06-01,8元留存5张3,4,3.0,1.0,1
96,145432939,2026-06-09,2026-06-13,留存6张3元,4,3.0,1.0,1
97,145755447,2026-08-04,2026-08-10,10元留存5张3,6,1.0,3.0,1
98,1459381870621,2026-05-13,2026-05-18,10元留存5张3,5,2.0,4.0,1


In [6]:
# 查重复入组的用户 完单记录
query = """
with base as (
    select
        dt,
        pid,
        case
            when flowtree_group_name = 'DS空白'
             and group_name_eve = 'control_group' then 0
            when group_name_eve = '留存6张3元' then 1
            when group_name_eve = '8元留存5张3' then 2
            when group_name_eve = '留存6张5元' then 3
            when group_name_eve = '10元留存5张3' then 4
        end as treatment_id,
        case
            when flowtree_group_name = 'DS空白'
             and group_name_eve = 'control_group' then 'DS空白'
            else group_name_eve
        end as treatment_name,
        coalesce(finish_cnt_7d, 0) as finish_cnt_7d,
        coalesce(send_success_cnt, 0) as send_success_cnt,
        coalesce(outcall_success_cnt, 0) as outcall_success_cnt,
        count(*) over (
            partition by
                pid,
                dt,
                flowtree_group_name,
                group_name_eve
        ) as same_day_row_cnt
    from dm.ut_alg_flowback_msg_outcall_label_v2_di
    where dt between '2026-07-09' and '2026-08-21'
      and dt not in (
          '2026-03-26',
          '2026-03-27',
          '2026-04-08',
          '2026-08-02',
          '2026-08-08',
          '2026-08-09'
      )
      and new_exp = '迭代流量'
      and pid is not null
      and (
          group_name_eve in (
              '留存6张3元',
              '8元留存5张3',
              '留存6张5元',
              '10元留存5张3'
          )
          or (
              flowtree_group_name = 'DS空白'
              and group_name_eve = 'control_group'
          )
      )
),

ordered_rows as (
    select
        b.*,
        lag(dt) over (
            partition by pid
            order by dt, treatment_id
        ) as previous_dt
    from base b
),

cycle_marked as (
    select
        b.*,
        case
            when previous_dt is null then 1
            when datediff(
                to_date(dt),
                to_date(previous_dt)
            ) >= 7 then 1
            else 0
        end as is_new_cycle
    from ordered_rows b
),

cycle_numbered as (
    select
        b.*,
        sum(is_new_cycle) over (
            partition by pid
            order by dt, treatment_id
            rows between unbounded preceding and current row
        ) as cycle_no
    from cycle_marked b
),

cycle_stats as (
    select
        b.*,
        min(dt) over (
            partition by pid, cycle_no
        ) as cycle_start_dt,
        max(dt) over (
            partition by pid, cycle_no
        ) as cycle_end_dt,
        count(*) over (
            partition by pid, cycle_no
        ) as entry_cnt,
        sum(
            case
                when finish_cnt_7d > 0 then 1
                else 0
            end
        ) over (
            partition by pid, cycle_no
        ) as positive_entry_cnt
    from cycle_numbered b
),

ranked_rows as (
    select
        b.*,
        row_number() over (
            partition by pid, cycle_no
            order by
                case
                    when finish_cnt_7d > 0 then 0
                    else 1
                end,
                finish_cnt_7d desc,
                dt desc
        ) as keep_rank
    from cycle_stats b
)

select
    pid,
    cycle_no,
    cycle_start_dt,
    cycle_end_dt,
    treatment_id,
    treatment_name,
    entry_cnt,
    positive_entry_cnt,
    case
        when positive_entry_cnt = 0 then 'all_negative'
        when positive_entry_cnt = 1 then 'one_positive'
        else 'multiple_positive'
    end as case_type,
    dt as entry_dt,
    finish_cnt_7d,
    send_success_cnt,
    outcall_success_cnt,
    same_day_row_cnt,
    case
        when keep_rank = 1 then 1
        else 0
    end as would_keep_flag
from ranked_rows
where entry_cnt > 1
  and cycle_start_dt between '2026-07-16' and '2026-08-21'
order by
    case
        when positive_entry_cnt > 1 then 0
        when positive_entry_cnt = 1 then 1
        else 2
    end,
    pid,
    cycle_no,
    entry_dt
limit 500000
"""

repeat_case_pdf = spark.sql(query).toPandas()
repeat_case_pdf

,pid,cycle_no,cycle_start_dt,cycle_end_dt,treatment_id,treatment_name,entry_cnt,positive_entry_cnt,case_type,entry_dt,finish_cnt_7d,send_success_cnt,outcall_success_cnt,same_day_row_cnt,would_keep_flag
0,1016407531520,1,2026-08-01,2026-08-04,3,留存6张5元,3,2,multiple_positive,2026-08-01,4.0,1.0,1.0,1,1
1,1016407531520,1,2026-08-01,2026-08-04,3,留存6张5元,3,2,multiple_positive,2026-08-03,0.0,0.0,0.0,1,0
2,1016407531520,1,2026-08-01,2026-08-04,3,留存6张5元,3,2,multiple_positive,2026-08-04,2.0,0.0,0.0,1,0
3,101861463,2,2026-07-22,2026-07-28,1,留存6张3元,6,2,multiple_positive,2026-07-22,1.0,1.0,0.0,1,0
4,101861463,2,2026-07-22,2026-07-28,1,留存6张3元,6,2,multiple_positive,2026-07-23,0.0,0.0,0.0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
499995,17592399335312,1,2026-07-17,2026-07-18,0,DS空白,2,0,all_negative,2026-07-17,0.0,0.0,0.0,1,0
499996,17592399335312,1,2026-07-17,2026-07-18,0,DS空白,2,0,all_negative,2026-07-18,0.0,0.0,0.0,1,1
499997,17592399335312,3,2026-08-05,2026-08-07,0,DS空白,2,0,all_negative,2026-08-05,0.0,0.0,0.0,1,0
499998,17592399335312,3,2026-08-05,2026-08-07,0,DS空白,2,0,all_negative,2026-08-07,0.0,0.0,0.0,1,1


In [10]:
# 查看典型的 0、0、x
# repeat_case_pdf[
#    repeat_case_pdf["case_type"] == "one_positive"
# ]

from IPython.display import display

one_positive_pdf = repeat_case_pdf[
    repeat_case_pdf["case_type"] == "one_positive"
].sort_values(
    ["pid", "cycle_no", "entry_dt"]
)

# 取前100条记录
result_100 = one_positive_pdf.head(100)

# 完整显示100行和所有列
with pd.option_context(
    "display.max_rows", 200,
    "display.max_columns", None,
    "display.width", None,
    "display.max_colwidth", None
):
    display(result_100)


,pid,cycle_no,cycle_start_dt,cycle_end_dt,treatment_id,treatment_name,entry_cnt,positive_entry_cnt,case_type,entry_dt,finish_cnt_7d,send_success_cnt,outcall_success_cnt,same_day_row_cnt,would_keep_flag
3289,100027189,3,2026-08-14,2026-08-20,0,DS空白,4,1,one_positive,2026-08-14,0.0,0.0,0.0,1,0
3290,100027189,3,2026-08-14,2026-08-20,0,DS空白,4,1,one_positive,2026-08-17,0.0,0.0,0.0,1,0
3291,100027189,3,2026-08-14,2026-08-20,0,DS空白,4,1,one_positive,2026-08-19,0.0,0.0,0.0,1,0
3292,100027189,3,2026-08-14,2026-08-20,0,DS空白,4,1,one_positive,2026-08-20,2.0,0.0,0.0,1,1
3293,100030829,2,2026-07-24,2026-07-27,1,留存6张3元,2,1,one_positive,2026-07-24,0.0,0.0,0.0,1,0
3294,100030829,2,2026-07-24,2026-07-27,1,留存6张3元,2,1,one_positive,2026-07-27,1.0,0.0,0.0,1,1
3295,100030829,3,2026-08-06,2026-08-07,1,留存6张3元,2,1,one_positive,2026-08-06,3.0,1.0,0.0,1,1
3296,100030829,3,2026-08-06,2026-08-07,1,留存6张3元,2,1,one_positive,2026-08-07,0.0,0.0,0.0,1,0
3297,1000857473024,1,2026-07-17,2026-07-18,1,留存6张3元,2,1,one_positive,2026-07-17,3.0,1.0,1.0,1,1
3298,1000857473024,1,2026-07-17,2026-07-18,1,留存6张3元,2,1,one_positive,2026-07-18,0.0,0.0,0.0,1,0


In [12]:
# 查看一个周期有多条正记录
repeat_case_pdf[
    repeat_case_pdf["case_type"] == "multiple_positive"
]



,pid,cycle_no,cycle_start_dt,cycle_end_dt,treatment_id,treatment_name,entry_cnt,positive_entry_cnt,case_type,entry_dt,finish_cnt_7d,send_success_cnt,outcall_success_cnt,same_day_row_cnt,would_keep_flag
0,1016407531520,1,2026-08-01,2026-08-04,3,留存6张5元,3,2,multiple_positive,2026-08-01,4.0,1.0,1.0,1,1
1,1016407531520,1,2026-08-01,2026-08-04,3,留存6张5元,3,2,multiple_positive,2026-08-03,0.0,0.0,0.0,1,0
2,1016407531520,1,2026-08-01,2026-08-04,3,留存6张5元,3,2,multiple_positive,2026-08-04,2.0,0.0,0.0,1,0
3,101861463,2,2026-07-22,2026-07-28,1,留存6张3元,6,2,multiple_positive,2026-07-22,1.0,1.0,0.0,1,0
4,101861463,2,2026-07-22,2026-07-28,1,留存6张3元,6,2,multiple_positive,2026-07-23,0.0,0.0,0.0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3284,978907365376,2,2026-08-11,2026-08-13,4,10元留存5张3,2,2,multiple_positive,2026-08-13,2.0,0.0,0.0,1,1
3285,998283485186,1,2026-07-18,2026-08-01,3,留存6张5元,4,2,multiple_positive,2026-07-18,0.0,1.0,0.0,1,0
3286,998283485186,1,2026-07-18,2026-08-01,3,留存6张5元,4,2,multiple_positive,2026-07-24,0.0,0.0,0.0,1,0
3287,998283485186,1,2026-07-18,2026-08-01,3,留存6张5元,4,2,multiple_positive,2026-07-26,2.0,1.0,0.0,1,0


In [11]:
multi_positive_pdf = repeat_case_pdf[
    repeat_case_pdf["case_type"] == "multiple_positive"
].sort_values(
    ["pid", "cycle_no", "entry_dt"]
)

# 取前100条记录
result_100 = multi_positive_pdf.head(100)

# 完整显示100行和所有列
with pd.option_context(
    "display.max_rows", 200,
    "display.max_columns", None,
    "display.width", None,
    "display.max_colwidth", None
):
    display(result_100)


,pid,cycle_no,cycle_start_dt,cycle_end_dt,treatment_id,treatment_name,entry_cnt,positive_entry_cnt,case_type,entry_dt,finish_cnt_7d,send_success_cnt,outcall_success_cnt,same_day_row_cnt,would_keep_flag
0,1016407531520,1,2026-08-01,2026-08-04,3,留存6张5元,3,2,multiple_positive,2026-08-01,4.0,1.0,1.0,1,1
1,1016407531520,1,2026-08-01,2026-08-04,3,留存6张5元,3,2,multiple_positive,2026-08-03,0.0,0.0,0.0,1,0
2,1016407531520,1,2026-08-01,2026-08-04,3,留存6张5元,3,2,multiple_positive,2026-08-04,2.0,0.0,0.0,1,0
3,101861463,2,2026-07-22,2026-07-28,1,留存6张3元,6,2,multiple_positive,2026-07-22,1.0,1.0,0.0,1,0
4,101861463,2,2026-07-22,2026-07-28,1,留存6张3元,6,2,multiple_positive,2026-07-23,0.0,0.0,0.0,1,0
5,101861463,2,2026-07-22,2026-07-28,1,留存6张3元,6,2,multiple_positive,2026-07-24,0.0,0.0,0.0,1,0
6,101861463,2,2026-07-22,2026-07-28,1,留存6张3元,6,2,multiple_positive,2026-07-25,0.0,0.0,0.0,1,0
7,101861463,2,2026-07-22,2026-07-28,1,留存6张3元,6,2,multiple_positive,2026-07-27,0.0,0.0,0.0,1,0
8,101861463,2,2026-07-22,2026-07-28,1,留存6张3元,6,2,multiple_positive,2026-07-28,1.0,0.0,0.0,1,1
9,103064592,1,2026-07-16,2026-07-20,2,8元留存5张3,2,2,multiple_positive,2026-07-16,2.0,1.0,0.0,1,0


In [13]:
import pandas as pd

multi_pdf = repeat_case_pdf[
    repeat_case_pdf["case_type"] == "multiple_positive"
].copy()

multi_pdf["cycle_start_dt"] = pd.to_datetime(
    multi_pdf["cycle_start_dt"]
)
multi_pdf["cycle_end_dt"] = pd.to_datetime(
    multi_pdf["cycle_end_dt"]
)

multi_pdf["cycle_span_days"] = (
    multi_pdf["cycle_end_dt"]
    - multi_pdf["cycle_start_dt"]
).dt.days

multi_summary = (
    multi_pdf
    .groupby(["pid", "cycle_no"], as_index=False)
    .agg(
        cycle_start_dt=("cycle_start_dt", "min"),
        cycle_end_dt=("cycle_end_dt", "max"),
        cycle_span_days=("cycle_span_days", "max"),
        entry_cnt=("entry_cnt", "max"),
        positive_entry_cnt=("positive_entry_cnt", "max"),
        max_same_day_row_cnt=("same_day_row_cnt", "max"),
        finish_cnt_sum=("finish_cnt_7d", "sum")
    )
)

multi_summary

,pid,cycle_no,cycle_start_dt,cycle_end_dt,cycle_span_days,entry_cnt,positive_entry_cnt,max_same_day_row_cnt,finish_cnt_sum
0,1016407531520,1,2026-08-01,2026-08-04,3,3,2,1,6.0
1,101861463,2,2026-07-22,2026-07-28,6,6,2,1,2.0
2,103064592,1,2026-07-16,2026-07-20,4,2,2,1,4.0
3,1038407241730,1,2026-07-16,2026-07-18,2,2,2,1,4.0
4,1051194298369,2,2026-07-25,2026-07-27,2,2,2,1,3.0
...,...,...,...,...,...,...,...,...,...
826,771835432962,1,2026-07-22,2026-07-28,6,2,2,1,2.0
827,904508276737,1,2026-07-26,2026-07-29,3,3,2,1,4.0
828,919289536514,1,2026-07-23,2026-07-26,3,4,2,1,6.0
829,978907365376,2,2026-08-11,2026-08-13,2,2,2,1,4.0


In [15]:
# 周期跨度超过7天：优先视为链式合并问题
multi_summary[
    multi_summary["cycle_span_days"] >= 7
]




,pid,cycle_no,cycle_start_dt,cycle_end_dt,cycle_span_days,entry_cnt,positive_entry_cnt,max_same_day_row_cnt,finish_cnt_sum
6,106790792,2,2026-07-19,2026-08-05,17,9,2,1,6.0
12,1109766184960,2,2026-07-23,2026-08-06,14,8,2,1,3.0
14,1124851449856,1,2026-07-19,2026-08-18,30,14,2,1,3.0
16,116616976,2,2026-07-23,2026-08-07,15,7,2,1,3.0
25,1272790122498,1,2026-07-31,2026-08-20,20,18,2,1,36.0
...,...,...,...,...,...,...,...,...,...
801,70369036444895,1,2026-07-16,2026-07-27,11,9,2,1,9.0
806,70369070084053,2,2026-08-06,2026-08-14,8,5,2,1,6.0
813,70369104059333,1,2026-07-26,2026-08-18,23,13,2,1,10.0
825,743836557313,1,2026-07-18,2026-08-01,14,8,2,1,5.0


In [16]:
# 同一天重复：优先检查源数据或join膨胀
multi_summary[
    multi_summary["max_same_day_row_cnt"] > 1
]


,pid,cycle_no,cycle_start_dt,cycle_end_dt,cycle_span_days,entry_cnt,positive_entry_cnt,max_same_day_row_cnt,finish_cnt_sum


In [17]:
# 周期跨度正常、同一天也不重复：
# 才是应该重点分析的真实多正case
multi_summary[
    (multi_summary["cycle_span_days"] < 7)
    & (multi_summary["max_same_day_row_cnt"] == 1)
]

,pid,cycle_no,cycle_start_dt,cycle_end_dt,cycle_span_days,entry_cnt,positive_entry_cnt,max_same_day_row_cnt,finish_cnt_sum
0,1016407531520,1,2026-08-01,2026-08-04,3,3,2,1,6.0
1,101861463,2,2026-07-22,2026-07-28,6,6,2,1,2.0
2,103064592,1,2026-07-16,2026-07-20,4,2,2,1,4.0
3,1038407241730,1,2026-07-16,2026-07-18,2,2,2,1,4.0
4,1051194298369,2,2026-07-25,2026-07-27,2,2,2,1,3.0
...,...,...,...,...,...,...,...,...,...
824,740164964355,2,2026-08-10,2026-08-12,2,3,2,1,3.0
826,771835432962,1,2026-07-22,2026-07-28,6,2,2,1,2.0
827,904508276737,1,2026-07-26,2026-07-29,3,3,2,1,4.0
828,919289536514,1,2026-07-23,2026-07-26,3,4,2,1,6.0


In [ ]:
# 查看一个周期全部为负
repeat_case_pdf[
    repeat_case_pdf["case_type"] == "all_negative"
]